# Aula 00 — Visão Geral do LangGraph (para dev clássico)

**Para quem:** você programa software determinístico (função, variável, `if/else`) e o LangGraph parece magia.
Não é. É um fluxograma com variável compartilhada + alguns nomes novos.

**Objetivo deste notebook:** ao final você deve conseguir dizer com suas palavras:
o que é Estado, Nó, Aresta, Aresta Condicional, `compile` e `invoke` — e por que um `if/else` simples não basta quando entra LLM.

**Como usar:** rode as 3 células de código em ordem. Tudo offline, sem chave, sem LLM.
O espelho executável está em `src/aula00_visao_geral.py` (`python src/aula00_visao_geral.py`).

## 1. Analogia: a lanchonete (você já sabe isso)

Pense num pedido como um **dict que vai sendo preenchido**:

```
pedido = {"itens": [], "com_sede": True}
receber_pedido(pedido)   # adiciona "hamburguer"
escolher_bebida(pedido)  # se com_sede: "suco", senão "água"
```

Tradução para o vocabulário LangGraph (decore esta tabela, ela volta o curso inteiro):

| Você já conhece | No LangGraph chama | O que é de verdade |
|---|---|---|
| dict compartilhado | **Estado (State)** | o `TypedDict` que todos os nós leem |
| função `f(pedido)->pedido` | **Nó (Node)** | função `f(state)->update` (devolve SÓ a mudança) |
| chamada em sequência | **Aresta (Edge)** | ordem fixa: `A -> B` |
| `if/else` | **Aresta condicional** | função que retorna o nome do próximo nó |
| montar o fluxograma | **`compile()`** | transforma o rascunho em executável |
| rodar com um input | **`invoke(input)`** | executa do `START` ao `END` e devolve o estado final |

A única diferença real: no código clássico a função devolve o dict inteiro;
no LangGraph o nó devolve **só o que mudou** (ex: `{"itens": [...]}`) e o framework mescla.

In [ ]:
# CÉLULA 1 — versão CLÁSSICA (confirme que é trivial para você)
def receber_pedido(pedido: dict) -> dict:
    pedido['itens'].append('hamburguer')
    return pedido

def escolher_bebida(pedido: dict) -> dict:
    if pedido.get('com_sede'):
        pedido['itens'].append('suco')
    else:
        pedido['itens'].append('água')
    return pedido

pedido = {'itens': [], 'com_sede': True}
pedido = receber_pedido(pedido)
pedido = escolher_bebida(pedido)
print(pedido['itens'])  # esperado: ['hamburguer', 'suco']
assert pedido['itens'] == ['hamburguer', 'suco']
print('OK — isso é tudo que o grafo vai fazer, só que organizado.')

## 2. Por que o `if/else` puro quebra? (os 4 desafios)

Enquanto é lanchonete, `if/else` basta. O problema aparece com 4 desafios trazidos pela IA (nada a ver com ingrediente do lanche — são desafios de engenharia):

Desafio 1 — **o LLM decide o caminho em runtime.** Você não sabe antes se o cliente quer fazer um pedido, pedir a conta ou perguntar se tem opção vegetariana — quem escolhe é o modelo, lendo texto livre. O `if` vira “pergunte ao LLM para onde ir”.
Desafio 2 — **ferramentas (tools).** O LLM pode pedir `multiplicar(3,4)` no meio da frase. Alguém precisa executar o Python, devolver o `12` e deixar o LLM continuar. Isso é um **ciclo**, não uma reta.
Desafio 3 — **ciclos.** Agente ReAct = `pensar → agir → observar → pensar…` até terminar. `if/else` encadeado não expressa “volte para trás N vezes”.
Desafio 4 — **memória entre turnos.** Cada `invoke` é stateless como HTTP. Sem um “banco de sessão” (`checkpointer`), o bot esquece tudo na próxima mensagem.

O LangGraph existe para desenhar **fluxos com ciclos, ferramentas e memória** sem virar spaghetti de `if`.

```
Clássico (reta):   receber -> bebida -> FIM

Agente (ciclo):   assistant -> precisa de tool? --sim--> tools
                       ^                                    |
                       └──────── continua aqui ──────────────┘
```

## 3. LangChain vs LangGraph (não confunda)

| | LangChain (Chain) | LangGraph (Graph) |
|---|---|---|
| Formato | Cano reto: passo 1 → 2 → 3 | Fluxograma: com bifurcação e loop |
| Decisão | fixa no código | pode ser do LLM em runtime |
| Estado | passa de mão em mão | mora num dict compartilhado |
| Quando usar | pipeline fixo (ex: resumir texto) | agente que decide e usa ferramentas |

Regra prática: se você consegue desenhar como reta, use chain. Se precisa de **“volte”, “escolha em runtime” ou “lembre entre mensagens”**, use grafo.

In [ ]:
# CÉLULA 2 — o MESMO lanche, agora em LangGraph (sem LLM!)
# Repare: o resultado é idêntico ao da célula 1. Só mudou a organização.
from typing import TypedDict
from langgraph.graph import START, END, StateGraph

class Pedido(TypedDict):
    itens: list
    com_sede: bool

def no_receber(state: Pedido) -> dict:
    return {'itens': state['itens'] + ['hamburguer']}

def no_bebida_com_sede(state: Pedido) -> dict:
    return {'itens': state['itens'] + ['suco']}

def no_bebida_sem_sede(state: Pedido) -> dict:
    return {'itens': state['itens'] + ['água']}

def decide_bebida(state: Pedido) -> str:
    return 'com_sede' if state['com_sede'] else 'sem_sede'  # seu if, com outro nome

builder = StateGraph(Pedido)
builder.add_node('receber', no_receber)
builder.add_node('com_sede', no_bebida_com_sede)
builder.add_node('sem_sede', no_bebida_sem_sede)
builder.add_edge(START, 'receber')
builder.add_conditional_edges('receber', decide_bebida)
builder.add_edge('com_sede', END)
builder.add_edge('sem_sede', END)
graph = builder.compile()

print(graph.invoke({'itens': [], 'com_sede': True})['itens'])
print(graph.invoke({'itens': [], 'com_sede': False})['itens'])

## 4. Dissecando a célula 2 linha a linha

- `class Pedido(TypedDict)` — o **contrato do dict**. Diz quais chaves existem. É só tipagem, não valida nada em runtime (validação de verdade vem na Aula 07 com Pydantic).
- `def no_receber(state) -> dict` — um **nó**. Recebe o estado inteiro, devolve **só a mudança**. Aqui usamos `state['itens'] + [...]` (lista nova) em vez de `append` — no grafo prefira não mutar o original.
- `builder = StateGraph(Pedido)` — cria o rascunho do fluxograma tipado com esse estado.
- `add_node('receber', no_receber)` — registra a função sob um nome (string). O nome é o “endereço” usado nas arestas.
- `add_edge(START, 'receber')` — aresta fixa: sempre começa aqui. `START`/`END` são marcadores especiais, não funções suas.
- `add_conditional_edges('receber', decide_bebida)` — após `receber`, chame `decide_bebida(state)`; o **valor retornado ('com_sede'/'sem_sede') escolhe o próximo nó**. É literalmente o `if` da célula 1.
- `builder.compile()` — “compila” o rascunho num objeto executável. Sem isso não roda.
- `graph.invoke({...})` — **uma execução completa** do `START` ao `END`. Sempre passe o estado inicial completo (aqui `itens` + `com_sede`). Retorna o estado final.

Detalhe que confunde dev clássico: `add_conditional_edges` com função que retorna string usa **o próprio retorno como nome do nó**. Por isso `'com_sede'` (retorno) precisa ser igual a `'com_sede'` (nome registrado). Se escrever diferente, dá erro de rota inexistente.

## 5. O que muda quando entra LLM, tool e memória? (só mapa mental, sem código)

**Variação A — com LLM:** um nó pode chamar o modelo. O estado vira lista de mensagens (`HumanMessage`, `AIMessage`). O LLM lê tudo e devolve texto ou um pedido de ferramenta.

**Variação B — com tool:** o “pedido de ferramenta” é um JSON (`tool_calls: [{name, args}]`). Um nó especial (`ToolNode`) executa a função Python de verdade e anexa o resultado (`ToolMessage(content='12')`). A aresta condicional (`tools_condition`) é o porteiro: `tem tool_calls? vá para tools, senão END`.

**Variação C — com memória:** `invoke` sozinho esquece tudo (como HTTP sem cookie). `compile(checkpointer=MemorySaver)` + `thread_id` = sessão. Mesma pergunta “multiplique por 2” resolve `14` se antes dissemos `isso=7`.

Guarde o filme: **Aulas 02–04 = fluxo; Aula 05 = ciclo; Aula 06 = sessão.**

## 6. Erros clássicos de quem vem do determinístico

1. **“O nó retorna o valor.”** Não — ele retorna um **update parcial**. Quem junta tudo é o framework. Se dois nós paralelos devolverem a mesma chave sem reducer, dá `InvalidUpdateError` (Aula 08).
2. **“`invoke` lembra da última chamada.”** Não lembra. Cada `invoke` é independente, a menos que compile com `checkpointer` + `thread_id` (Aula 06).
3. **“LLM é determinístico.”** Não é — `temperature` e sorteio mudam a saída. Nos estudos usamos `seed`/`Fake` para fixar; em produção, trate variação como normal.
4. **“Paralelo mantém ordem.”** Fan-out (`A -> B + C`) executa junto; sem reducer a mesclagem é ambígua. Ordem só existe em aresta sequencial.
5. **“Aresta condicional com `if` dentro do nó basta.”** O `if` dentro do nó não muda a rota — só a função de roteamento (retorno string) muda. Decisão de caminho sempre mora na aresta, não no nó.

In [ ]:
# CÉLULA 3 — desafio guiado (complete as lacunas e rode)
# Situação: a lanchonete agora tem SOBREMESA. Se com_fome=True, adiciona 'bolo'.
from typing import TypedDict
from langgraph.graph import START, END, StateGraph

class Pedido2(TypedDict):
    itens: list
    com_fome: bool

def no_base(state: Pedido2) -> dict:
    return {'itens': state['itens'] + ['hamburguer']}

def no_bolo(state: Pedido2) -> dict:
    return {'itens': state['itens'] + ['bolo']}  # TODO: você escreveu este

def no_sem_bolo(state: Pedido2) -> dict:
    return {'itens': state['itens']}  # nada a adicionar

def decide(state: Pedido2) -> str:
    return 'fome' if state['com_fome'] else 'sem_fome'  # seu if como roteador

b = StateGraph(Pedido2)
b.add_node('base', no_base)
b.add_node('fome', no_bolo)
b.add_node('sem_fome', no_sem_bolo)
b.add_edge(START, 'base')
b.add_conditional_edges('base', decide)
b.add_edge('fome', END)
b.add_edge('sem_fome', END)
g = b.compile()

r1 = g.invoke({'itens': [], 'com_fome': True})['itens']
r2 = g.invoke({'itens': [], 'com_fome': False})['itens']
print(r1, '|', r2)
assert r1 == ['hamburguer', 'bolo'], r1
assert r2 == ['hamburguer'], r2
print('OK — você montou um grafo com bifurcação sozinho.')

## 7. Checklist + próxima aula

**Você entendeu se consegue responder sem olhar:**
- [ ] O que mora no Estado e quem pode ler/escrever nele?
- [ ] O que um nó recebe e o que ele devolve?
- [ ] Diferença entre aresta fixa e condicional?
- [ ] Para que servem `compile` e `invoke`?
- [ ] Por que `if/else` puro não expressa ciclo de agente nem tool?

**Próxima aula (01 — Setup):** venv, `requirements`, `.env` + `getpass` sem vazar chave, Groq vs OpenAI do curso, `invoke vs stream`, `Human/AIMessage` como structs, teste de sanidade com e sem chave.

Se algo acima ficou nebuloso, me diga qual frase travou — a Aula 01 parte deste ponto.